In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## အဆင့် ၁: ဖွဲ့စည်းပုံသတ်မှတ်ထားသော အထွက်များအတွက် Pydantic မော်ဒယ်များ သတ်မှတ်ခြင်း

ဤမော်ဒယ်များသည် အေးဂျင့်များ ပြန်လည်ပေးပို့မည့် **schema** ကို သတ်မှတ်သည်။ Pydantic ဖြင့် `response_format` ကို အသုံးပြုခြင်းအားဖြင့် အောက်ဖော်ပြပါ အချက်များလုံခြုံစေသည်-
- ✅ အမျိုးအစား-လုံခြုံသော ဒေတာ ဖော်ထုတ်မှု
- ✅ အလိုအလျောက် အတည်ပြုမှု
- ✅ လွတ်လပ်သော စာသားဖြင့်ဖြေကြားမှုမှ parsing အမှား မရှိခြင်း
- ✅ လယ်ဝါများအပေါ် အခြေခံ၍ စနစ်တကျ လမ်းညွှန်မှု လွယ်ကူစွာလုပ်ဆောင်နိုင်ခြင်း


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: ဟိုတယ် အဝေးထိုးကိရိယာကို ဖန်တီးခြင်း

ဤကိရိယာသည် **availability_agent** အနေဖြင့် အခန်းများ ရနိုင်မှုကို စစ်ဆေးရန် ခေါ်ဆိုမည့် ကိရိယာဖြစ်သည်။ `@ai_function` decorator ကို အသုံးပြု၍ -
- Python function ကို AI ခေါ်ဆိုနိုင်မည့် ကိရိယာဖြစ်အောင် ပြောင်းလဲခြင်း
- LLM အတွက် JSON schema ကို အလိုအလျောက် ထုတ်ပေးခြင်း
- parameter သတ်မှတ်ချက် စစ်ဆေးခြင်းကို ကိုင်တွယ်ပေးခြင်း
- လမ်းညွှန်များမှ အလိုအလျောက် ခေါ်ဆိုနိုင်စေရေး

ဒီ demonstration အတွက် -
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → အခန်းများ ရရှိနိုင် ✅
- **အခြားမြို့များအားလုံး** → အခန်းမရှိခြင်း ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## အဆင့် 3: လမ်းကြောင်းသတ်မှတ်ရန် အခြေအနေ ဖန်ရှင်များ ကို သတ်မှတ်ခြင်း

ဒီဖန်ရှင်တွေက agent ရဲ့ ပြန်လည်ဖြေကြားမှုကို စစ်ဆေးပြီး workflow မှာ ဘယ်လမ်းကြောင်းကို သွားမလဲ ဆိုတာဆုံးဖြတ်ပေးပါတယ်။

**အရေးကြီး နမူနာ:**
1. မက်ဆေ့ချျ်က `AgentExecutorResponse` ဖြစ်မရှိ စစ်ဆေးပါ
2. ဖွဲ့စည်းထားသော output (Pydantic မော်ဒယ်) ကို ဗျည်းထုတ်ပါ
3. လမ်းညွှန်မှု အတွက် `True` သို့မဟုတ် `False` ကို ပြန်ပေးပါ

workflow ကဒီအခြေအနေတွေကို **အကွက်များ**ပေါ်မှာ စမ်းသပ်ကာ နောက်ပိုင်း အသုံးပြုမယ့် executor ကိုဆုံးဖြတ်ပါလိမ့်မယ်။


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## အဆင့် ၄: မှန်ကန်သော ပြသမှု ရှက်ကွပ်တည်ဆောက်ခြင်း

ရှက်ကွပ်များသည် မဟာဗျူဟာအတွင်းပြုလုပ်သော ပြုပြင်ပြောင်းလဲမှုများ သို့မဟုတ် ဘက်ထွက်သက်ရောက်မှုများဖြစ်သည်။ နောက်ဆုံးရလဒ်ကို ပြသပေးသော မှန်ကန်သော ရှက်ကွပ်ကို ဖန်တီးရန် `@executor` decorator ကို အသုံးပြုသည်။

**အဓိကခံယူချက်များ**
- `@executor(id="...")` - မဟာဗျူဟာ ရှက်ကွပ်အဖြစ် function ကို မှတ်ပုံတင်သည်
- `WorkflowContext[Never, str]` - နေရာထည့်/ထွက် အတွက် အမျိုးအစား အကြံပြုချက်များ
- `ctx.yield_output(...)` - နောက်ဆုံး မဟာဗျူဟာရလဒ်ကို yield ပေးသည်


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Step 5: ပတ်ဝန်းကျင်ပြောင်းလဲမှုများကို လုပ်ဆောင်ခြင်း

LLM client ကို ပြင်ဆင်ပါ။ ဤဥပမာသည် အောက်ပါအရာများနှင့် အတူ လုပ်ဆောင်သည် -
- **GitHub Models** (GitHub token ဖြင့် အခမဲ့ အဆင့်)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ခြေလှမ်း ၆: ဖွဲ့စည်းထားသော ထုတ်လွှင့်ချက်များရှိ AI ကိုယ်စားလှယ်များ ဖန်တီးခြင်း

ကျွန်ုပ်တို့သည် **ဝန္ထမ်း သုံးဦးကို** ဖန်တီးပြီး၊ တစ်ဦးချင်းစီကို `AgentExecutor` ဖြင့် အထုပ်ဖုံးထားသည်။

1. **availability_agent** - ကိရိယာကို အသုံးပြု၍ ဟိုတယ် ရရှိနိုင်မှု စစ်ဆေးသည်
2. **alternative_agent** - အခန်း မရှိသောအခါ တခြားမြို့များကို အကြံပြုသည်
3. **booking_agent** - အခန်း ရရှိနိုင်သောအခါ မှာယူရန် အားပေးသည်

**အဓိက လက္ခဏာများ:**
- `tools=[hotel_booking]` - ကိုယ်စားလှယ်ထံသို့ ကိရိယာကို ပံ့ပိုးပေးသည်
- `response_format=PydanticModel` - ဖော်မြူလာထားသော JSON ထုတ်လွှင့်ချက်ကို အတင်းအကြပ် အတည်ပြုသည်
- `AgentExecutor(..., id="...")` - workflow အသုံးပြုမှုအတွက် ကိုယ်စားလှယ်ကို အထုပ်ဖုံးသည်


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## အဆင့် ၇: အခြေအနေအရရှိသော အဆက်များဖြင့် Workflow တည်ဆောက်ခြင်း

ယခု `WorkflowBuilder` ကို အသုံးပြုပြီး အခြေအနေအရရှိသော အဆက်များဖြင့် ဂရပ်ဖ်ကို တည်ဆောက်မှာ ဖြစ်ပါတယ်။

**Workflow ဖွဲ့စည်းမှု:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**အဓိက နည်းလမ်းများ:**
- `.set_start_executor(...)` - စတင်အသုံးပြုသူကိုသတ်မှတ်သည်
- `.add_edge(from, to, condition=...)` - အခြေအနေအရရှိသော အဆက်ကို ထည့်သည်
- `.build()` - Workflow ကို အဆုံးသတ်သည်


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 8: တတ်စမ်းမှုကိစ္စ ၁ ကို အကောင်အထည်ဖော်ပါ - နေရာများမရှိသော မြို့ (ပါရီ)

အခန်းမရှိခြင်း လမ်းကြောင်းကို စမ်းသပ်ကြည့်ရန် ပါရီတွင် ဟိုတယ်များအား တောင်းဆိုခြင်းဖြင့် စမ်းသပ်ကြည့်ပါ (အဲဒီမှာ ကျွန်ုပ်တို့၏ စမ်းသပ်မှုတွင် အခန်းမရှိပါ။)။


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: Run Test Case 2 - City WITH Availability (Stockholm)

ခုနောက်မှာတော့ **availability** လမ်းကြောင်းကို စမ်းသပ်ကြပါစို့၊ Stockholm မှ ဟိုတယ်များကို တောင်းဆိုခြင်းဖြင့် (ကျွန်ုပ်တို့၏ သရုပ်ပြတွင် အခန်းများရှိသည်)။


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## အဓိကယူဆချက်များနှင့် နောက်တစ်ဆင့်များ

### ✅ သင်သင်ယူခဲ့သောအချက်များ

1. **WorkflowBuilder ပုံစံ**
   - `.set_start_executor()` ကို ဝင်ကြားချက်အနေဖြင့် သတ်မှတ်ရန် အသုံးပြုပါ
   - `.add_edge(from, to, condition=...)` ကို အခြေအနေအရ လမ်းကြောင်းသတ်မှတ်ရန် အသုံးပြုပါ
   - `.build()` ကို ခေါ်ပြီး workflow ကို အဆုံးသတ်ပါ

2. **အခြေအနေအရ လမ်းကြောင်းရွေးချယ်ခြင်း**
   - Condition functions တို့သည် `AgentExecutorResponse` ကို စစ်ဆေးကြသည်
   - ဖွဲ့စည်းထားသော output များကို ဖော်ထုတ်၍ လမ်းကြောင်းဆုံးဖြတ်ချက်များပြုလုပ်သည်
   - လမ်းကြောင်းတစ်ခုကို ဖွင့်ရန် `True` ပြန်ပေးပြီး၊ မလမ်းကြောင်းရောက်ရန် `False` ပြန်ပေးသည်

3. **ကိရိယာပေါင်းစည်းခြင်း**
   - Python function များကို AI tool များအဖြစ် ပြောင်းရန် `@ai_function` ကို အသုံးပြုပါ
   - Agents များသည် လိုအပ်သောအခါ အလိုအလျောက် ကိရိယာများကို ခေါ်သုံးသည်
   - ကိရိယာများသည် agents များထုတ်ယူနိုင်သော JSON ကို ပြန်ပေးသည်

4. **ဖွဲ့စည်းထားသော output များ**
   - Pydantic မော်ဒယ်များကို data များကို စနစ်တကျ ထုတ်ယူရန် အသုံးပြုပါ
   - Agents ဖန်တီးစဉ် `response_format=MyModel` ကို သတ်မှတ်ပါ
   - `Model.model_validate_json()` ဖြင့် အဖြေများကို စစ်ဆေးပါ

5. **စိတ်ကြိုက် Executors များ**
   - Workflow အစိတ်အပိုင်းများ ဖန်တီးရန် `@executor(id="...")` ကို အသုံးပြုပါ
   - Executors များသည် data များပြောင်းလဲမှု သို့မဟုတ် ဘက်ဆိုင်ရာလုပ်ဆောင်ချက်များ ပြုလုပ်နိုင်သည်
   - Workflow ရလဒ်များ ထုတ်လုပ်ရန် `ctx.yield_output()` ကို အသုံးပြုပါ

### 🚀 အကောင်အထည်ဖော်နိုင်သော နေရာများ

- **ခရီးသွား စီစဉ်ခြင်း**: ရရှိနိုင်မှုစစ်ဆေး၊ ဒုတိယရွေးချယ်စရာများ အကြံပြု၊ ရွေးချယ်မှုများ နှိုင်းယှဉ်
- **ဖောက်သည် ဝန်ဆောင်မှု**: ပြဿနာအမျိုးအစား၊ စိတ်ခံစားမှု၊ ဦးစားပေးမှုအရ လမ်းကြောင်းသတ်မှတ်ခြင်း
- **လျှော့စျေးစီးပွားရေး**: ပစ္စည်းလက်ကျန်စစ်ဆေး၊ ဒုတိယရွေးချယ်စရာများ အကြံပြု၊ အမှာစာများ ဆောင်ရွက်ခြင်း
- **ထိန်းချုပ်ရေးအကြောင်းအရာ**: အန္တရာယ်ရှိမှု အဆင့်များ၊ အသုံးပြုသူ အမှတ်အသားများအရ လမ်းကြောင်းသတ်မှတ်ခြင်း
- **အတည်ပြုမှု Workflow များ**: ပမာဏ၊ အသုံးပြုသူ အခန်းကဏ္ဍ၊ အန္တရာယ်အဆင့်အရ လမ်းကြောင်းရွေးချယ်ခြင်း
- **အဆင့်မြင့် ပြုပြင်ထိန်းသိမ်းမှု**: ဒေတာအရည်အသွေး၊ ပြည့်စုံမှုပေါ်ပေါက်ခြင်းအတိုင်း လမ်းကြောင်းသတ်မှတ်ခြင်း

### 📚 နောက်တစ်ဆင့်များ

- ပိုမိုရှုပ်ထွေးသော အခြေအနေများကို ထည့်သွင်းခြင်း (Criteria များစွာ)
- Workflow အခြေအနေစီမံမှုဖြင့် Loop များ ဆောင်ရွက်ခြင်း
- သုံးစွဲနိုင်သော အစိတ်အပိုင်းများအတွက် Sub-workflows ထည့်သွင်းခြင်း
- အမှန်တကယ် API များနှင့် ပေါင်းစည်းခြင်း (ဟိုတယ် 예약, စတော့အိတ်စနစ်များ)
- အမှားကိုင်တွယ်ခြင်းနှင့် အစွန်းရောက်လမ်းကြောင်းများ ထည့်သွင်းခြင်း
- Built-in visualization ကိရိယာများဖြင့် workflows ကို မြင်သာစေရန် ပြသခြင်း


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
